In [ ]:
!pip install qiskit pennylane torch gymnasium numpy matplotlib

# Module 2: Multi-Agent Reinforcement Learning (MARL) Agent Training
This notebook implements the Multi-Agent Deep Deterministic Policy Gradient / Actor-Critic architecture in PyTorch to optimize quantum circuit operations.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Actor Network for Sub-Agent (Single-qubit / Two-qubit Gate Optimizers)
class QASActor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QASActor, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
            nn.Softmax(dim=-1)
        )

    def forward(self, state):
        return self.fc(state)

# Critic Network for Global State Evaluation
class QASCritic(nn.Module):
    def __init__(self, state_dim):
        super(QASCritic, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, state):
        return self.fc(state)

# Multi-Agent Trainer Module
class MARLQuantumOptimizer:
    def __init__(self, state_dim=30, action_dim=3, lr=1e-3):
        self.actor = QASActor(state_dim, action_dim)
        self.critic = QASCritic(state_dim)
        self.actor_opt = optim.Adam(self.actor.parameters(), lr=lr)
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=lr)

    def train_step(self, state, action, reward, next_state, done):
        state_t = torch.FloatTensor(state).view(1, -1)
        next_state_t = torch.FloatTensor(next_state).view(1, -1)
        reward_t = torch.FloatTensor([reward])

        # Value estimation
        value = self.critic(state_t)
        next_value = self.critic(next_state_t)
        target_value = reward_t + (0.99 * next_value * (1 - int(done)))

        # Critic loss
        critic_loss = nn.MSELoss()(value, target_value.detach())
        self.critic_opt.zero_grad()
        critic_loss.backward()
        self.critic_opt.step()

        # Actor loss
        probs = self.actor(state_t)
        log_prob = torch.log(probs[0, action] + 1e-8)
        advantage = (target_value - value).detach()
        actor_loss = -log_prob * advantage

        self.actor_opt.zero_grad()
        actor_loss.backward()
        self.actor_opt.step()

        return critic_loss.item(), actor_loss.item()

if __name__ == "__main__":
    optimizer = MARLQuantumOptimizer(state_dim=30, action_dim=3)
    dummy_state = np.random.rand(30)
    dummy_next = np.random.rand(30)
    c_loss, a_loss = optimizer.train_step(dummy_state, 1, 0.85, dummy_next, False)
    print(f"[+] MARL Agent Training Step Completed -> Critic Loss: {c_loss:.4f}, Actor Loss: {a_loss:.4f}")